In [ ]:
import os, re
from pathlib import Path
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torchaudio

if os.name == "posix":
    AUDIO_ROOT = Path("/home/wanted-1/PotenupWorkspace/aug-project5/jin_sup/audios")
    SAVE_DIR   = Path("/home/wanted-1/PotenupWorkspace/aug-project5/jin_sup/model")
else:
    AUDIO_ROOT = Path(r"C:\PythonProject\aug-08month_project5\jin_sup\audios")
    SAVE_DIR   = Path(r"C:\PythonProject\aug-08month_project5\jin_sup\model")

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("CUDA available:", torch.cuda.is_available())

RANGES = [
    ((1, 50),   "Happiness"),
    ((51, 100), "Surprise"),
    ((101,150), "Neutral"),
    ((151,200), "Fear"),
    ((201,250), "Disgust"),
    ((251,300), "Anger"),
    ((301,350), "Sadness"),
]
CLASSES = [name for _, name in RANGES]
label2idx = {name:i for i,name in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

def score_to_label(score: int) -> str:
    for (low, high), label in RANGES:
        if low <= score <= high:
            return label
    raise ValueError(f"점수 범위 밖: {score}")

def parse_label_from_name(path: str) -> int:
    stem = Path(path).stem  # "090-039"
    m = re.match(r"^\d+-(\d+)$", stem)
    if not m:
        raise ValueError(f"파일명 파싱 실패: {path}")
    return int(m.group(1))

def parse_label_idx(path: str) -> int:
    score = parse_label_from_name(path)
    name  = score_to_label(score)
    return label2idx[name]

CUDA available: True


In [52]:
all_audio_paths = sorted([str(p) for p in AUDIO_ROOT.rglob("*.wav") if p.is_file()])
y_all = list(map(parse_label_idx, all_audio_paths))

X_train, X_test, y_train, y_test = train_test_split(
    all_audio_paths, y_all, test_size=0.2, stratify=y_all, random_state=42
)
print(f"train/test: {len(X_train)} / {len(X_test)}")

train/test: 8280 / 2071


In [53]:
SR = 16000
N_FFT = 1024
HOP   = 256               
N_MELS = 64               
TOP_DB = 70               
FMIN_MEL, FMAX_MEL = 50.0, 8000.0
FMIN_F0,  FMAX_F0  = 50.0, 800.0     
TARGET = 224


In [ ]:


# GPT 코드들.. 아래는 다 지피티 꺼 (코드 파악중)
mel_spec = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
    f_min=FMIN_MEL, f_max=FMAX_MEL, power=2.0, norm="slaney", mel_scale="htk"
).to(device)
to_db = torchaudio.transforms.AmplitudeToDB(top_db=TOP_DB).to(device)

def _avg_pool_2d(x: torch.Tensor, k_freq: int, k_time: int) -> torch.Tensor:
    # x: [1, H, T] 또는 [H, T]
    if x.dim() == 2: x = x.unsqueeze(0)
    x = F.avg_pool2d(x, kernel_size=(k_freq, k_time), stride=(k_freq, k_time), ceil_mode=True)
    return x

def _resize_to(x: torch.Tensor, H: int, T: int) -> torch.Tensor:
    # x: [1, h, t]
    x = F.interpolate(x.unsqueeze(0), size=(H,T), mode="bilinear", align_corners=False).squeeze(0)
    return x  

def _f0_to_band(y_pos: np.ndarray, H: int, sigma: float = 1.5) -> np.ndarray:
    # y_pos: [T] (0..H-1 위치), 가우시안 띠
    yy = np.arange(H, dtype=np.float32)[:, None]
    dist2 = (yy - y_pos[None,:])**2
    band = np.exp(-dist2/(2*sigma**2))
    return (band - band.min()) / (band.max()-band.min() + 1e-8)

def wav_to_prosody_tensor(path: str) -> torch.Tensor:
    # 0) 로드
    y, sr = librosa.load(path, sr=SR, mono=True)
    y_t = torch.from_numpy(y).float().to(device)

    # 1) 멜 (저해상도 + 평활화)
    S = mel_spec(y_t)                            
    Sdb = to_db(S).clamp_(-TOP_DB, 0.0)
    Smel = (Sdb + TOP_DB) / TOP_DB               
    Smel = Smel.unsqueeze(0)                     

    #   음소 내용 희석: 주파수/시간 풀링으로 블러 + 다시 원래 크기로 리사이즈
    #   (k_freq, k_time)는 데이터에 맞춰 조정 가능. 값이 클수록 단어 정보 더 흐림.
    k_freq, k_time = 4, 4
    Smel_low = _avg_pool_2d(Smel, k_freq, k_time)          
    Smel_smooth = _resize_to(Smel_low, Smel.shape[1], Smel.shape[2])  
    ch_mel = Smel_smooth  

    # 2) 피치(F0) → 띠
    f0, vflag, vconf = librosa.pyin(y, fmin=FMIN_F0, fmax=FMAX_F0, sr=sr, frame_length=N_FFT, hop_length=HOP)
    T0 = len(f0)
    valid = ~np.isnan(f0)
    if valid.sum() < 3:
        ch_pitch = torch.zeros_like(ch_mel)
        vprob = torch.zeros(Smel.shape[-1], dtype=torch.float32, device=device)
    else:
        idx = np.arange(T0)
        f0i = np.interp(idx, idx[valid], f0[valid]).astype(np.float32)
        ylog = np.log(np.clip(f0i, FMIN_F0, FMAX_F0))
        y_pos = (ylog - np.log(FMIN_F0)) / (np.log(FMAX_F0)-np.log(FMIN_F0)) * (N_MELS-1)

        # 시간 정렬: 멜 T와 동일하게
        t_src = np.linspace(0,1,T0, dtype=np.float32)
        t_dst = np.linspace(0,1,Smel.shape[-1], dtype=np.float32)
        y_pos = np.interp(t_dst, t_src, y_pos)

        band = _f0_to_band(y_pos, N_MELS, sigma=1.5)       # [H,T] 0..1
        ch_pitch = torch.from_numpy(band).unsqueeze(0).float().to(device)
        vprob = np.nan_to_num(vconf, nan=0.0)
        vprob = np.interp(t_dst, t_src, vprob).astype(np.float32)

    # 3) RMS 에너지 → 밴드
    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP)[0]  # [T_rms]
    
    # 멜 T와 정렬
    t_src2 = np.linspace(0,1,len(rms), dtype=np.float32)
    t_dst2 = np.linspace(0,1,Smel.shape[-1], dtype=np.float32)
    rms_t = np.interp(t_dst2, t_src2, rms)

    # 0..1 정규화 (강세/리듬만)
    rms_t = (rms_t - rms_t.min()) / (rms_t.max() - rms_t.min() + 1e-8)
    ch_rms = torch.from_numpy(np.tile(rms_t[None,:], (N_MELS,1))).unsqueeze(0).float().to(device)  # [1,H,T]

    # 4) 스택 & 리사이즈 → [3, TARGET, TARGET], 스케일 [-1,1]
    x = torch.cat([ch_mel, ch_pitch, ch_rms], dim=0)   # [3,H,T], 0..1
    x = F.interpolate(x.unsqueeze(0), size=(TARGET,TARGET), mode="bilinear", align_corners=False).squeeze(0)
    x = x*2 - 1  
    return x  

In [55]:
class ProsodyDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = list(paths); self.labels = list(labels)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return wav_to_prosody_tensor(self.paths[i]), self.labels[i]

BATCH = 32
NUM_WORKERS = 0
train_dl = DataLoader(ProsodyDataset(X_train, y_train), batch_size=BATCH, shuffle=True,
                      num_workers=NUM_WORKERS)
test_dl  = DataLoader(ProsodyDataset(X_test,  y_test),  batch_size=BATCH, shuffle=False,
                      num_workers=NUM_WORKERS)


In [56]:
import sys
sys.path.append(r"C:\PythonProject\aug-08month_project5")

from data_model.BuildModel import BuildModel
from data_model.ModelType import ModelType
import torch
import torchvision.models as M


CLASSES = list(map(lambda x: x[1], RANGES))
label2idx = {name:i  for i,name in enumerate(CLASSES)}

buildModel = BuildModel(ModelType.CONVNEXT_SMALL, len(RANGES), M.ConvNeXt_Small_Weights.IMAGENET1K_V1)
model = buildModel.model
model.to(device)
device


c:\PythonProject\aug-08month_project5\.venv\lib\site-packages\torchvision\models\_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(


'cuda:0'

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from datetime import datetime
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

writer = SummaryWriter(log_dir="runs/convnext_experiment1")

lr =  3e-4
criterion = nn.CrossEntropyLoss()
optimz = torch.optim.AdamW(model.parameters(), lr= lr, weight_decay=1e-5)
steps_per_epoch = len(train_dl)
EPOCHS = 5
total_steps = EPOCHS * steps_per_epoch
warmup_steps = max(100, int(0.05 * total_steps)) 

sched_warmup = LinearLR(optimz, start_factor=0.01, total_iters=warmup_steps) 
T_max = max(1, total_steps - warmup_steps)  
sched_cosine = CosineAnnealingLR(optimz, T_max=T_max, eta_min=lr * 0.01)
scheduler = SequentialLR(optimz, [sched_warmup, sched_cosine], milestones=[warmup_steps])

# 아래는 GPT 꺼.. 함부로 쓰기엔 애매하다.
# steps_per_epoch = len(train_dl)
# total_steps = max(1, EPOCHS * steps_per_epoch)
# warmup_steps = max(100, int(0.05 * total_steps))
# sched_warmup = torch.optim.lr_scheduler.LinearLR(optimz, start_factor=0.01, total_iters=warmup_steps)
# T_max = max(1, total_steps - warmup_steps)
# sched_cos = torch.optim.lr_scheduler.CosineAnnealingLR(optimz, T_max=T_max, eta_min=lr*0.01)
# scheduler = torch.optim.lr_scheduler.SequentialLR(optimz, [sched_warmup, sched_cos], milestones=[warmup_steps])

def acc_fn(logits, y):
    return (logits.argmax(1) == y).float().mean().item()

writer = SummaryWriter(log_dir="runs/")
step = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    
    for train, label in tqdm(train_dl):
        try: 
            optimz.zero_grad()
            train, label = train.to(device), label.to(device)
            
            logits = model(train)             
            loss = criterion(logits, label)   

            loss.backward()
            optimz.step()  
            scheduler.step()
            #텐서 보드 기록
            writer.add_scalar("Loss/train", loss.item(), step)
            step += 1
            
        except Exception as e:
            print(f"❌ Error in training loop (epoch={epoch}): {e}")
            raise  

timestamp = datetime.now().strftime("%m-%d_%H-%M-%S")
filename = f"result_{timestamp}"
torch.save(model.state_dict(), f"model/model_{buildModel.weights}___{timestamp}.pth")


  2%|▏         | 4/259 [02:27<2:37:03, 36.96s/it]


SystemError: CPUDispatcher(<function _viterbi at 0x0000018D0F335870>) returned a result with an exception set